# Faza 4 - wariant C (tylko syntetyczne, simple prompts, BEZ real)

C odpowiada na pytanie: czy syntetyki same wystarcza? Trening na 1200 synth (10 klas x 120, simple SD 1.5 prompts, BEZ LoRA), bez ani jednego prawdziwego zdjecia. Test set ten sam co A/B/D/E (988 real).

Roznica wzgledem notebook 08 (E):
- pull `variant_D.zip` (simple prompts) zamiast `variant_E.zip` (LoRA mix)
- pattern run_all: `C_*.yaml`
- config C ma `use_real: false` (real_samples = [])

## 1. Repo + Pets

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q timm==1.0.11 transformers==4.46.3 huggingface_hub==0.26.2 PyYAML==6.0.2

In [ ]:
from pathlib import Path
PETS = Path('data/raw/oxford-iiit-pet/images')
if not (PETS.exists() and any(PETS.iterdir())):
    !python scripts/download_pets.py

## 2. HF auth + pull variant_D + manifest

In [ ]:
from huggingface_hub import login, hf_hub_download, whoami
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('brak HF_TOKEN - dodaj w Colab Secrets')
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in as:', whoami()['name'])

HF_REPO_ID = 'micwuj/dlicv-synth'

In [ ]:
import zipfile, shutil
VARIANT_D = Path('data/synthetic/variant_D')
MANIFEST = VARIANT_D / 'manifest_kept.csv'
sample_png = VARIANT_D / 'Ragdoll' / 'Ragdoll_0000.png'

if not sample_png.exists():
    zip_path = hf_hub_download(
        repo_id=HF_REPO_ID, repo_type='dataset',
        filename='variant_D.zip', local_dir='data/synthetic',
    )
    VARIANT_D.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(VARIANT_D)
    # auto-flatten na wypadek zagniezdzonego variant_D/
    nested = VARIANT_D / 'variant_D'
    if nested.exists() and nested.is_dir():
        for d in nested.iterdir():
            tgt = VARIANT_D / d.name
            if tgt.exists():
                shutil.rmtree(tgt)
            shutil.move(str(d), str(tgt))
        nested.rmdir()
    print(f'rozpakowane pngs: {sum(1 for _ in VARIANT_D.rglob("*.png"))}')
else:
    print(f'variant_D juz rozpakowane ({sum(1 for _ in VARIANT_D.rglob("*.png"))} pngs).')

# Manifest dociagamy osobno (bezpiecznik na wypadek pustego w zipie)
hf_hub_download(
    repo_id=HF_REPO_ID, repo_type='dataset',
    filename='variant_D/manifest_kept.csv', local_dir='data/synthetic',
)
with open(MANIFEST) as f:
    n_rows = sum(1 for _ in f) - 1
print(f'manifest: {n_rows} rows')
assert n_rows > 2000, f'manifest looks broken: {n_rows} rows'
assert sample_png.exists(), f'missing sample png: {sample_png}'

## 3. Petla treningu C

In [ ]:
!python scripts/generate_configs.py

In [ ]:
!python -u scripts/run_all.py --pattern 'C_*.yaml' --overrides configs/colab.yaml --skip-existing

## 4. Final dump + aggregate

In [ ]:
from google.colab import files
shutil.make_archive('/content/outputs_C_all', 'zip', '.', 'outputs')
files.download('/content/outputs_C_all.zip')